# Water Annotator Example — WaterMap & WaterFLAP

This notebook demonstrates how to:

1. **Validate & annotate** WaterMap hydration sites using the `WaterMapAnnotator`
2. **Annotate** WaterFLAP hydration sites using the `WaterFLAPAnnotator`
3. **Inspect** and **compare** results from both tools
4. **Visualise** both sets of water sites together in 3D (WaterMap as spheres, WaterFLAP as stars)

**Inputs:**
- `8irv_wm_data.csv` + `8irv_wm_waters.pdb` — WaterMap output
- `WFapo_8IRV.pdb` — WaterFLAP output (dG in B-factor column)

## 1. Imports and Setup

In [27]:
from pathlib import Path
from collections import Counter

import py3Dmol

from water_annotator import (
    WaterMapCSVStandardiser,
    validate_watermap_csv,
    EXPECTED_COLUMNS,
    WaterMapAnnotator,
    WaterFLAPAnnotator,
    WaterCategory,
)

from water_annotator.base import classify_water

In [28]:
# Shared structure files
data_dir = Path("../tests/test_data/drd5_example")
result_dir = Path("../result")
result_dir.mkdir(exist_ok=True)

protein_pdb = data_dir / "8IRV.pdb"
ligand_sdf = data_dir / "8IRV_R5F_lig.sdf"

# WaterMap inputs
wm_csv_path = data_dir / "8irv_wm_data.csv"
wm_pdb_path = data_dir / "8irv_wm_waters.pdb"

# WaterFLAP input
wf_pdb_path = data_dir / "WFapo_8IRV.pdb"

---
## 2. WaterMap — Validate & Annotate

The `WaterMapCSVStandardiser` checks that the CSV contains all required columns:

`Site, Entry ID, Occupancy, Overlap, dH, -TdS, dG, #HB(WW), #HB(PW), #HB(LW)`

In [29]:
# Validate the WaterMap CSV
print("Expected columns:", EXPECTED_COLUMNS)
standardiser = WaterMapCSVStandardiser()
found_columns = standardiser.validate(wm_csv_path)
print("Validation passed! Found columns:", found_columns)

Expected columns: frozenset({'dH', 'Site', '#HB(WW)', 'Entry ID', 'Occupancy', '#HB(LW)', 'dG', 'Overlap', '#HB(PW)', '-TdS'})
Validation passed! Found columns: ['Site', 'Entry ID', 'Occupancy', 'Overlap', 'dH', '-TdS', 'dG', '#HB(WW)', '#HB(PW)', '#HB(LW)']


In [30]:
# Create annotator and write classified PDB
wm_annotator = WaterMapAnnotator(csv_path=wm_csv_path, water_pdb_path=wm_pdb_path)
wm_output = wm_annotator.annotate(result_dir / "annotated_watermap.pdb", title="8IRV WaterMap hydration sites")
print(f"WaterMap annotated PDB written to: {wm_output}")

WaterMap annotated PDB written to: ../result/annotated_watermap.pdb


In [31]:
# Inspect WaterMap sites
wm_sites = wm_annotator.sites
print(f"WaterMap — {len(wm_sites)} hydration sites")
for site in wm_sites[:5]:
    print(f"  Site {site.site_id}: dG={site.dG:+.2f} kcal/mol, "
          f"coords=({site.x:.3f}, {site.y:.3f}, {site.z:.3f})")

wm_counts = Counter(classify_water(s.dG) for s in wm_sites)
print("\nWaterMap category counts:")
for cat in WaterCategory:
    print(f"  {cat.name:<15s} ({cat.value}): {wm_counts.get(cat, 0)}")

WaterMap — 82 hydration sites
  Site 1: dG=+6.68 kcal/mol, coords=(-14.358, -12.808, 40.384)
  Site 2: dG=+6.01 kcal/mol, coords=(-13.663, -2.574, 46.296)
  Site 3: dG=+2.76 kcal/mol, coords=(-0.137, -13.244, 57.938)
  Site 4: dG=-1.82 kcal/mol, coords=(-17.794, -10.613, 56.056)
  Site 5: dG=+2.61 kcal/mol, coords=(-0.860, -6.858, 61.893)

WaterMap category counts:
  VERY_UNHAPPY    (WVU): 14
  UNHAPPY         (WUN): 17
  BULK_LIKE       (WBL): 40
  HAPPY           (WHP): 11


---
## 3. WaterFLAP — Annotate

The `WaterFLAPAnnotator` reads the raw WaterFLAP PDB directly — coordinates and dG values are extracted from the HETATM records (dG is stored in the B-factor column).

In [32]:
# Create annotator and write classified PDB
wf_annotator = WaterFLAPAnnotator(pdb_path=wf_pdb_path)
wf_output = wf_annotator.annotate(result_dir / "annotated_waterflap.pdb", title="8IRV WaterFLAP hydration sites")
print(f"WaterFLAP annotated PDB written to: {wf_output}")

WaterFLAP annotated PDB written to: ../result/annotated_waterflap.pdb


In [33]:
# Inspect WaterFLAP sites
wf_sites = wf_annotator.sites
print(f"WaterFLAP — {len(wf_sites)} hydration sites")
for site in wf_sites[:5]:
    print(f"  Site {site.site_id}: dG={site.dG:+.2f} kcal/mol, "
          f"coords=({site.x:.3f}, {site.y:.3f}, {site.z:.3f})")

wf_counts = Counter(classify_water(s.dG) for s in wf_sites)
print("\nWaterFLAP category counts:")
for cat in WaterCategory:
    print(f"  {cat.name:<15s} ({cat.value}): {wf_counts.get(cat, 0)}")

WaterFLAP — 66 hydration sites
  Site 1: dG=-2.70 kcal/mol, coords=(-7.480, -10.590, 60.000)
  Site 2: dG=-1.40 kcal/mol, coords=(-13.710, -12.110, 58.010)
  Site 3: dG=-0.60 kcal/mol, coords=(-13.140, -2.380, 46.720)
  Site 4: dG=-0.90 kcal/mol, coords=(-8.980, -14.610, 58.590)
  Site 6: dG=-1.20 kcal/mol, coords=(-9.647, -10.000, 55.490)

WaterFLAP category counts:
  VERY_UNHAPPY    (WVU): 4
  UNHAPPY         (WUN): 13
  BULK_LIKE       (WBL): 40
  HAPPY           (WHP): 9


---
## 4. Compare category distributions

In [34]:
print(f"{'Category':<20s} {'WaterMap':>10s} {'WaterFLAP':>10s}")
print("-" * 42)
for cat in WaterCategory:
    wm_n = wm_counts.get(cat, 0)
    wf_n = wf_counts.get(cat, 0)
    print(f"{cat.name:<15s} ({cat.value}) {wm_n:>10d} {wf_n:>10d}")
print("-" * 42)
print(f"{'Total':<20s} {len(wm_sites):>10d} {len(wf_sites):>10d}")

Category               WaterMap  WaterFLAP
------------------------------------------
VERY_UNHAPPY    (WVU)         14          4
UNHAPPY         (WUN)         17         13
BULK_LIKE       (WBL)         40         40
HAPPY           (WHP)         11          9
------------------------------------------
Total                        82         66


---
## 5. 3D Visualisation — WaterMap only

Colour key:
- **Red** — Very unhappy (WVU)
- **Yellow** — Unhappy (WUN)
- **Grey** — Bulk-like (WBL)
- **Blue** — Happy (WHP)

In [35]:
CATEGORY_COLOURS = {
    WaterCategory.VERY_UNHAPPY: "red",
    WaterCategory.UNHAPPY: "yellow",
    WaterCategory.BULK_LIKE: "grey",
    WaterCategory.HAPPY: "blue",
}

viewer_wm = py3Dmol.view(width=1000, height=800)
viewer_wm.addModel(protein_pdb.read_text(), "pdb")
viewer_wm.setStyle({"model": 0}, {"cartoon": {"color": "lightgrey", "opacity": 0.7}})
viewer_wm.addModel(ligand_sdf.read_text(), "sdf")
viewer_wm.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})

for site in wm_sites:
    colour = CATEGORY_COLOURS[classify_water(site.dG)]
    viewer_wm.addSphere({
        "center": {"x": site.x, "y": site.y, "z": site.z},
        "radius": 0.5, "color": colour, "opacity": 0.8,
    })

viewer_wm.zoomTo({"model": 1})
viewer_wm.zoom(0.7)
viewer_wm.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---
## 6. 3D Visualisation — WaterFLAP only

In [36]:
viewer_wf = py3Dmol.view(width=1000, height=800)
viewer_wf.addModel(protein_pdb.read_text(), "pdb")
viewer_wf.setStyle({"model": 0}, {"cartoon": {"color": "lightgrey", "opacity": 0.7}})
viewer_wf.addModel(ligand_sdf.read_text(), "sdf")
viewer_wf.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})

for site in wf_sites:
    colour = CATEGORY_COLOURS[classify_water(site.dG)]
    viewer_wf.addSphere({
        "center": {"x": site.x, "y": site.y, "z": site.z},
        "radius": 0.5, "color": colour, "opacity": 0.8,
    })

viewer_wf.zoomTo({"model": 1})
viewer_wf.zoom(0.7)
viewer_wf.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---
## 7. Combined 3D Visualisation — WaterMap + WaterFLAP

Both sets of water sites are shown together to compare predictions:
- **WaterMap** sites → **spheres**
- **WaterFLAP** sites → **stars** (rendered as smaller cross-shaped markers)

Same colour encoding for both: Red (WVU), Yellow (WUN), Grey (WBL), Blue (WHP).

In [37]:
viewer_combined = py3Dmol.view(width=1000, height=800)

# Protein + ligand
viewer_combined.addModel(protein_pdb.read_text(), "pdb")
viewer_combined.setStyle({"model": 0}, {"cartoon": {"color": "lightgrey", "opacity": 0.7}})
viewer_combined.addModel(ligand_sdf.read_text(), "sdf")
viewer_combined.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})

# WaterMap sites — spheres (solid)
for site in wm_sites:
    colour = CATEGORY_COLOURS[classify_water(site.dG)]
    viewer_combined.addSphere({
        "center": {"x": site.x, "y": site.y, "z": site.z},
        "radius": 0.5, "color": colour, "opacity": 0.85,
    })

# WaterFLAP sites — stars (6 thin cylinders radiating from centre)
STAR_ARM = 0.45  # half-length of each arm
STAR_RADIUS = 0.06  # cylinder thickness
STAR_DIRS = [
    (1, 0, 0), (0, 1, 0), (0, 0, 1),
    (0.707, 0.707, 0), (0.707, 0, 0.707), (0, 0.707, 0.707),
]

for site in wf_sites:
    colour = CATEGORY_COLOURS[classify_water(site.dG)]
    for dx, dy, dz in STAR_DIRS:
        viewer_combined.addCylinder({
            "start": {
                "x": site.x - dx * STAR_ARM,
                "y": site.y - dy * STAR_ARM,
                "z": site.z - dz * STAR_ARM,
            },
            "end": {
                "x": site.x + dx * STAR_ARM,
                "y": site.y + dy * STAR_ARM,
                "z": site.z + dz * STAR_ARM,
            },
            "radius": STAR_RADIUS,
            "color": colour,
            "opacity": 0.85,
            "fromCap": True,
            "toCap": True,
        })

viewer_combined.zoomTo({"model": 1})
viewer_combined.zoom(0.7)
viewer_combined.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---
## 8. Custom thresholds

Override the default classification thresholds for both annotators.

Combined 3D visualisation with custom thresholds follows below (WaterMap = spheres, WaterFLAP = stars).

In [38]:
annotator_kwargs = dict(very_unhappy_threshold=4.0, unhappy_threshold=2.5, happy_threshold=-1.5)
classify_kwargs = dict(very_unhappy=4.0, unhappy=2.5, happy=-1.5)

custom_wm = WaterMapAnnotator(csv_path=wm_csv_path, water_pdb_path=wm_pdb_path, **annotator_kwargs)
custom_wm.annotate(result_dir / "annotated_watermap_custom.pdb", title="8IRV WaterMap (custom)")

custom_wf = WaterFLAPAnnotator(pdb_path=wf_pdb_path, **annotator_kwargs)
custom_wf.annotate(result_dir / "annotated_waterflap_custom.pdb", title="8IRV WaterFLAP (custom)")

custom_classify = lambda dG: classify_water(dG, **classify_kwargs)

print(f"{'Category':<20s} {'WaterMap':>10s} {'WaterFLAP':>10s}")
print("-" * 42)
for cat in WaterCategory:
    wm_n = sum(1 for s in custom_wm.sites if custom_classify(s.dG) == cat)
    wf_n = sum(1 for s in custom_wf.sites if custom_classify(s.dG) == cat)
    print(f"{cat.name:<15s} ({cat.value}) {wm_n:>10d} {wf_n:>10d}")

Category               WaterMap  WaterFLAP
------------------------------------------
VERY_UNHAPPY    (WVU)         12          3
UNHAPPY         (WUN)         16          6
BULK_LIKE       (WBL)         47         51
HAPPY           (WHP)          7          6


In [39]:
viewer_custom = py3Dmol.view(width=1000, height=800)

# Protein + ligand
viewer_custom.addModel(protein_pdb.read_text(), "pdb")
viewer_custom.setStyle({"model": 0}, {"cartoon": {"color": "lightgrey", "opacity": 0.7}})
viewer_custom.addModel(ligand_sdf.read_text(), "sdf")
viewer_custom.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})

# WaterMap sites — spheres (solid)
for site in custom_wm.sites:
    colour = CATEGORY_COLOURS[custom_classify(site.dG)]
    viewer_custom.addSphere({
        "center": {"x": site.x, "y": site.y, "z": site.z},
        "radius": 0.5, "color": colour, "opacity": 0.85,
    })

# WaterFLAP sites — stars (6 thin cylinders radiating from centre)
for site in custom_wf.sites:
    colour = CATEGORY_COLOURS[custom_classify(site.dG)]
    for dx, dy, dz in STAR_DIRS:
        viewer_custom.addCylinder({
            "start": {
                "x": site.x - dx * STAR_ARM,
                "y": site.y - dy * STAR_ARM,
                "z": site.z - dz * STAR_ARM,
            },
            "end": {
                "x": site.x + dx * STAR_ARM,
                "y": site.y + dy * STAR_ARM,
                "z": site.z + dz * STAR_ARM,
            },
            "radius": STAR_RADIUS,
            "color": colour,
            "opacity": 0.85,
            "fromCap": True,
            "toCap": True,
        })

viewer_custom.zoomTo({"model": 1})
viewer_custom.zoom(0.7)
viewer_custom.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.